# medaka-3d 完整流程批量运行（m3dfull）

把 `20260112_P_1_1_FCMC_medaka3d_full_chain.ipynb` 里验证过的流程批量跑到其他实验：
入口缓存 → c27 → c28 → c33 → c30 → c34 → 导出（λ = 0.767）→ 审阅包 `<tag>_m3dfull`。

- 逻辑在 `scripts/reconstruction_3d/medaka3d_batch.py`，这里只负责启动和看进度
- 每一步已完成就跳过，中断后重新启动即可接着跑；一个实验失败不影响其他实验
- 只用 CPU，不和 GPU 推理抢资源
- **K_1 的检测和其他实验不同**：K_1 用 medaka-3d 自己的 s16 分块（不限实例数），合并到 `s16_bottomup_3node_uncapped`，并和 c01 缓存逐帧核对；其他实验用 `--max_instances 6` 的 s14 `.slp`

## 步骤 1 — 参数和定位

- 输入：当前目录、`configs/paths.<用户名>.toml`、`../medaka-3d`
- `RUN_BATCH`：为 `True` 时步骤 2 才启动（防止“全部运行”误启动）
- 输出：`EXPERIMENTS`、`LOG_DIR`

In [1]:
import os
import subprocess
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for d in [start, *start.parents]:
        if (d / "pyproject.toml").exists() and (d / "scripts" / "common" / "local_paths.py").exists():
            return d
    raise FileNotFoundError(f"找不到仓库根目录（从 {start} 向上没有 pyproject.toml）")


REPO = find_repo_root(Path.cwd().resolve())
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import pandas as pd

from scripts.reconstruction_3d import medaka3d_batch as MB
from scripts.reconstruction_3d import medaka3d_chain as M3C

EXPERIMENTS = [
    "20251225_K_1_1_FCMC", "20251225_K_1_2_FCMC", "20251225_K_1_3_FCMC",
    "20260113_P_1_2_FCMC", "20260112_P_1_3_FCMC",
    "20260118_P_2_1_FCMC", "20260118_P_2_2_FCMC", "20260118_P_2_3_FCMC",
    "20260109_K_2_1_FCMC", "20260109_K_2_2_FCMC", "20260109_K_2_3_FCMC",
]
JOBS = 3
RUN_BATCH = False
LOG_DIR = M3C.m3d_paths(M3C.medaka3d_repo()).work_dir / "medaka3d_chain_logs"
print("仓库:", REPO)
print("日志:", LOG_DIR)

仓库: C:\Users\ReiterU\Documents\GitHub\medaka-mating-pipeline
日志: Z:\medaka\work\medaka3d_chain_logs


## 步骤 2 — 后台启动

- 独立进程，关掉 notebook 也继续跑
- 输出：`LOG_DIR/batch_<本次>.log`（总日志）、`LOG_DIR/<实验>/batch.log`（单个实验）、`LOG_DIR/<实验>/<步骤>.log`

In [2]:
if RUN_BATCH:
    log = open(LOG_DIR / "batch_notebook.log", "a", encoding="utf-8")
    flags = subprocess.CREATE_NEW_PROCESS_GROUP | subprocess.DETACHED_PROCESS if os.name == "nt" else 0
    p = subprocess.Popen([sys.executable, "-m", "scripts.reconstruction_3d.medaka3d_batch", *EXPERIMENTS, "--jobs", str(JOBS)],
                         stdout=log, stderr=subprocess.STDOUT, cwd=REPO, creationflags=flags,
                         env=dict(os.environ, PYTHONIOENCODING="utf-8", PYTHONUNBUFFERED="1"))
    print("已启动，PID", p.pid)
else:
    print("未启动：把步骤 1 的 RUN_BATCH 改成 True 再运行本格")

未启动：把步骤 1 的 RUN_BATCH 改成 True 再运行本格


## 步骤 3 — 进度

- 每个实验一行：检测来源、c27…c34 的状态（`ok` / `failed` / `running` / `missing`）、是否已导出、审阅包是否已生成
- 随时重跑本格

In [3]:
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 30)
PROGRESS = pd.DataFrame([MB.status_row(e) for e in EXPERIMENTS])
display(PROGRESS)
print("全部完成:", bool(PROGRESS["review"].all()))

,experiment,detections,c27,c28,c33,c30,c34,export,review
0,20251225_K_1_1_FCMC,s16_bottomup_3node_uncapped,ok,ok,ok,ok,ok,True,True
1,20251225_K_1_2_FCMC,s16_bottomup_3node_uncapped,ok,ok,ok,ok,ok,True,True
2,20251225_K_1_3_FCMC,s16_bottomup_3node_uncapped,ok,ok,ok,ok,ok,True,True
3,20260113_P_1_2_FCMC,s14_bottomup_3node_reviewed,ok,ok,ok,ok,ok,True,True
4,20260112_P_1_3_FCMC,s14_bottomup_3node_reviewed,ok,ok,ok,ok,ok,True,True
5,20260118_P_2_1_FCMC,s14_bottomup_3node_reviewed,ok,ok,ok,ok,ok,True,True
6,20260118_P_2_2_FCMC,s14_bottomup_3node_reviewed,ok,ok,ok,ok,ok,True,True
7,20260118_P_2_3_FCMC,s14_bottomup_3node_reviewed,ok,ok,ok,ok,ok,True,True
8,20260109_K_2_1_FCMC,s14_bottomup_3node_reviewed,ok,ok,ok,ok,ok,True,True
9,20260109_K_2_2_FCMC,s14_bottomup_3node_reviewed,ok,ok,ok,ok,ok,True,True


全部完成: True
